# 01 - Betimsel Analiz

2021+ temiz veri seti uzerinde oturum, harcama, tercih ve hava degiskenlerinin betimsel ozetleri.

**Veri:** `Veriler/oturum_hava_temiz_2021_sonrasi.csv` (120.749 kayit)

**Hedef metrikler:** `oturum_sure_dk`, `toplam_miktar`, `urun_sayisi`, `toplam_tutar_deflate_2021`


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.family"] = "DejaVu Sans"

PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "Veriler" / "oturum_hava_temiz_2021_sonrasi.csv"
OUTPUT_DIR = PROJECT_ROOT / "Outputs" / "Analizler" / "01_descriptive"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["oturum_sure_dk", "toplam_miktar", "urun_sayisi", "toplam_tutar_deflate_2021"]
MEVSIM_ORDER = ["Kış", "İlkbahar", "Yaz", "Sonbahar"]
OGUN_ORDER = ["Öğle", "İkindi", "Akşam", "Diğer"]

if not DATA_PATH.exists():
    raise FileNotFoundError("Once 00_veri_hazirlama_2021_sonrasi.ipynb calistirin.")

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["acilis_datetime", "kapama_datetime", "tarih", "merge_saati"],
)
print("Veri boyutu:", df.shape)
print("Tarih araligi:", df["tarih"].min().date(), "->", df["tarih"].max().date())


Veri boyutu: (120749, 66)
Tarih araligi: 2021-01-02 -> 2025-05-28


## 1. Genel betimsel istatistikler


In [2]:
genel = df[METRICS].describe(percentiles=[0.25, 0.5, 0.75]).T.round(3)
genel["cv_pct"] = (100 * genel["std"] / genel["mean"]).round(2)
genel.to_csv(OUTPUT_DIR / "genel_betimsel_istatistik.csv", encoding="utf-8-sig")
genel


,count,mean,std,min,25%,50%,75%,max,cv_pct
oturum_sure_dk,120749.0,33.013,33.991,1.00,15.00,25.000,38.000,240.000,102.96
toplam_miktar,120749.0,5.911,4.798,0.75,3.00,5.000,8.000,191.000,81.17
urun_sayisi,120749.0,3.406,2.046,1.00,2.00,3.000,4.000,25.000,60.07
toplam_tutar_deflate_2021,120749.0,172.856,153.575,0.00,74.96,130.494,221.835,5343.263,88.85


## 2. Masa grubu dagilimi ve KPI


In [3]:
masa_counts = (
    df["masa_grup"].value_counts(dropna=False)
    .rename_axis("masa_grup").reset_index(name="oturum_sayisi")
)
masa_counts["oran_yuzde"] = (100 * masa_counts["oturum_sayisi"] / len(df)).round(2)

masa_kpi = (
    df.groupby("masa_grup", observed=True)[METRICS]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)
masa_kpi.columns = ["_".join(c) for c in masa_kpi.columns]
masa_kpi = masa_kpi.reset_index()

masa_counts.to_csv(OUTPUT_DIR / "masa_grup_dagilimi.csv", index=False, encoding="utf-8-sig")
masa_kpi.to_csv(OUTPUT_DIR / "masa_grup_kpi.csv", index=False, encoding="utf-8-sig")

print("Masa grubu dagilimi:")
print(masa_counts.to_string(index=False))
masa_kpi.head()


Masa grubu dagilimi:
masa_grup  oturum_sayisi  oran_yuzde
 İç Salon          47052       38.97
    Bahçe          37622       31.16
    Paket          36075       29.88


,masa_grup,oturum_sure_dk_count,oturum_sure_dk_mean,oturum_sure_dk_median,oturum_sure_dk_std,toplam_miktar_count,toplam_miktar_mean,toplam_miktar_median,toplam_miktar_std,urun_sayisi_count,urun_sayisi_mean,urun_sayisi_median,urun_sayisi_std,toplam_tutar_deflate_2021_count,toplam_tutar_deflate_2021_mean,toplam_tutar_deflate_2021_median,toplam_tutar_deflate_2021_std
0,Bahçe,37622,30.193,27.0,18.022,37622,6.279,5.0,4.302,37622,3.790,3.0,1.969,37622,173.037,138.467,133.074
1,Paket,36075,43.781,24.0,55.031,36075,4.489,3.0,3.981,36075,2.220,2.0,1.130,36075,141.492,105.729,137.400
2,İç Salon,47052,27.013,24.0,15.882,47052,6.705,6.0,5.464,47052,4.008,3.0,2.259,47052,196.759,149.920,174.837


## 3. Zaman degiskenleri


In [4]:
def group_kpi(data, col, order=None):
    g = (
        data.groupby(col, observed=True)[METRICS]
        .agg(oturum_sayisi=("oturum_sure_dk", "size"), ort_sure=("oturum_sure_dk", "mean"),
             ort_miktar=("toplam_miktar", "mean"), ort_urun=("urun_sayisi", "mean"),
             ort_tutar_deflate=("toplam_tutar_deflate_2021", "mean"))
        .round(3).reset_index()
    )
    if order:
        g[col] = pd.Categorical(g[col], categories=order, ordered=True)
        g = g.sort_values(col)
    return g

zaman_tables = {
    "mevsim": group_kpi(df, "mevsim", MEVSIM_ORDER),
    "covid_donemi": group_kpi(df, "covid_donemi"),
    "gun_tipi": group_kpi(df, "gun_tipi"),
    "oglen_aksam": group_kpi(df, "oglen_aksam", OGUN_ORDER),
    "yil": group_kpi(df, "yil"),
}

for name, tbl in zaman_tables.items():
    tbl.to_csv(OUTPUT_DIR / f"zaman_{name}_kpi.csv", index=False, encoding="utf-8-sig")

zaman_tables["mevsim"]


,mevsim,oturum_sayisi,ort_sure,ort_miktar,ort_urun,ort_tutar_deflate
0,Kış,33740,33.287,5.888,3.335,171.907
3,İlkbahar,23826,32.588,5.912,3.393,189.162
2,Yaz,28727,32.589,5.829,3.424,166.355
1,Sonbahar,34456,33.393,6.000,3.469,167.931


## 4. Hava kategorileri


In [ ]:
hava_cols = [
    "sicaklik_aralik", "yagis_kategori", "ruzgar_seviyesi", "nem_grubu",
    "yagis_yogunlugu", "bulut_grubu",
]
hava_tables = {}
for col in hava_cols:
    if col not in df.columns:
        continue
    tbl = group_kpi(df, col)
    # Masa tercih payi
    pref = (
        df.groupby([col, "masa_grup"], observed=True).size()
        .groupby(level=0).apply(lambda s: 100 * s / s.sum())
        .reset_index(name="tercih_oran_yuzde").round(2)
    )
    tbl = tbl.merge(
        pref.pivot(index=col, columns="masa_grup", values="tercih_oran_yuzde").reset_index(),
        on=col, how="left",
    )
    hava_tables[col] = tbl
    tbl.to_csv(OUTPUT_DIR / f"hava_{col}_kpi.csv", index=False, encoding="utf-8-sig")

hava_tables["sicaklik_aralik"].head(10)


ValueError: cannot insert sicaklik_aralik, already exists

## 5. Gorsellestirmeler


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Masa grubu dagilimi
sns.barplot(data=masa_counts, x="masa_grup", y="oran_yuzde", ax=axes[0, 0], palette="Set2")
axes[0, 0].set_title("Masa Grubu Tercih Dagilimi (%)")
axes[0, 0].set_ylabel("Oran (%)")

# Mevsimsel oturum suresi
sns.barplot(data=zaman_tables["mevsim"], x="mevsim", y="ort_sure", ax=axes[0, 1], palette="coolwarm")
axes[0, 1].set_title("Mevsimlere Gore Ort. Oturum Suresi (dk)")
axes[0, 1].set_ylabel("Dakika")

# Mevsimsel bahce payi (masa tercihi)
mevsim_pref = (
    df.groupby(["mevsim", "masa_grup"], observed=True).size()
    .groupby(level=0).apply(lambda s: 100 * s / s.sum())
    .reset_index(name="oran")
)
mevsim_pref["mevsim"] = pd.Categorical(mevsim_pref["mevsim"], categories=MEVSIM_ORDER, ordered=True)
mevsim_pivot = mevsim_pref.pivot(index="mevsim", columns="masa_grup", values="oran").fillna(0)
mevsim_pivot.plot(kind="bar", stacked=True, ax=axes[1, 0], colormap="Set3")
axes[1, 0].set_title("Mevsim x Masa Grubu Tercih Payi (%)")
axes[1, 0].set_ylabel("Oran (%)")
axes[1, 0].legend(title="Masa", bbox_to_anchor=(1.02, 1))

# Yillik deflate tutar vs miktar
yil_tbl = zaman_tables["yil"]
ax2 = axes[1, 1]
ax2.plot(yil_tbl["yil"], yil_tbl["ort_tutar_deflate"], marker="o", label="Deflate tutar (2021 baz)")
ax2.set_ylabel("Ort. deflate tutar")
ax2.set_xlabel("Yil")
ax2b = ax2.twinx()
ax2b.plot(yil_tbl["yil"], yil_tbl["ort_miktar"], marker="s", color="orange", label="Ort. miktar")
ax2b.set_ylabel("Ort. miktar")
ax2.set_title("Yillik Deflate Tutar vs Miktar")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "betimsel_ozet_grafikleri.png", dpi=150, bbox_inches="tight")
plt.show()
print("Grafik kaydedildi:", OUTPUT_DIR / "betimsel_ozet_grafikleri.png")


In [ ]:
# Sicaklik x masa tercihi heatmap
pref_matrix = pd.crosstab(df["sicaklik_aralik"], df["masa_grup"], normalize="index") * 100
plt.figure(figsize=(9, 5))
sns.heatmap(pref_matrix.round(1), annot=True, fmt=".1f", cmap="YlGnBu", cbar_kws={"label": "%"})
plt.title("Sicaklik Araligi x Masa Grubu Tercih Payi")
plt.ylabel("Sicaklik araligi")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sicaklik_masa_tercih_heatmap.png", dpi=150)
plt.show()

# Hava kategorileri - oturum suresi boxplot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="yagis_yogunlugu", y="oturum_sure_dk", ax=axes[0], palette="Blues")
axes[0].set_title("Yagis Yogunlugu x Oturum Suresi")
axes[0].tick_params(axis="x", rotation=25)
sns.boxplot(data=df, x="ruzgar_seviyesi", y="oturum_sure_dk", ax=axes[1],
            order=["Sakin", "Hafif", "Orta", "Kuvvetli"], palette="Greens")
axes[1].set_title("Ruzgar Seviyesi x Oturum Suresi")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "hava_oturum_suresi_boxplot.png", dpi=150)
plt.show()


## 6. Ozet rapor


In [ ]:
# One-page executive summary
top_masa = masa_counts.iloc[0]
yaz = zaman_tables["mevsim"][zaman_tables["mevsim"]["mevsim"] == "Yaz"].iloc[0]
kis = zaman_tables["mevsim"][zaman_tables["mevsim"]["mevsim"] == "Kış"].iloc[0]

ozet = {
    "kayit_sayisi": int(len(df)),
    "en_yogun_masa_grubu": top_masa["masa_grup"],
    "en_yogun_masa_oran_pct": float(top_masa["oran_yuzde"]),
    "ort_oturum_sure_dk": round(float(df["oturum_sure_dk"].mean()), 2),
    "ort_toplam_miktar": round(float(df["toplam_miktar"].mean()), 2),
    "ort_tutar_deflate_2021": round(float(df["toplam_tutar_deflate_2021"].mean()), 2),
    "yaz_ort_sure_dk": float(yaz["ort_sure"]),
    "kis_ort_sure_dk": float(kis["ort_sure"]),
    "yaz_ort_miktar": float(yaz["ort_miktar"]),
    "kis_ort_miktar": float(kis["ort_miktar"]),
}

with open(OUTPUT_DIR / "betimsel_ozet.json", "w", encoding="utf-8") as f:
    json.dump(ozet, f, ensure_ascii=False, indent=2)

lines = [
    "BETIMSEL ANALIZ OZETI (2021+)",
    "=" * 50,
    f"Kayit: {ozet['kayit_sayisi']:,}",
    f"Baskin masa grubu: {ozet['en_yogun_masa_grubu']} (%{ozet['en_yogun_masa_oran_pct']})",
    f"Ort. oturum suresi: {ozet['ort_oturum_sure_dk']} dk",
    f"Ort. miktar: {ozet['ort_toplam_miktar']} | Deflate tutar: {ozet['ort_tutar_deflate_2021']} TL",
    f"Yaz vs Kis sure: {ozet['yaz_ort_sure_dk']} vs {ozet['kis_ort_sure_dk']} dk",
    f"Yaz vs Kis miktar: {ozet['yaz_ort_miktar']} vs {ozet['kis_ort_miktar']}",
]
text = chr(10).join(lines)
(OUTPUT_DIR / "betimsel_ozet.txt").write_text(text, encoding="utf-8")
print(text)
print("\nCikti klasoru:", OUTPUT_DIR.resolve())
